#### Loading de dados

In [15]:
import numpy as np
import pandas as pd
from lightweight_charts import JupyterChart

df = pd.read_parquet('/Users/marcelogarcia/Documents/Development/autobt/autobt/data/usatec2017_2025.parquet')

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
df['shdn'] = df.low.values - np.minimum(df.open.values, df.close.values)
df['shdn_percentile'] = np.abs(df['shdn']).rolling(window=10000).rank(pct=True)

df['shdn_percentile'].hist(bins=10, edgecolor='black', figsize=(8, 4))
plt.title('shdn_percentile Histogram')
plt.xlabel('shdn_percentile')
plt.ylabel('Count')
plt.tight_layout()
plt.show()



#### Funções de sinais

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

df['shdn_percentile'].hist(bins=10, edgecolor='black', figsize=(8, 4))
plt.title('shdn_percentile Histogram')
plt.xlabel('shdn_percentile')
plt.ylabel('Count')
plt.tight_layout()
plt.show()


In [ ]:
"""
================================================================================
CANDLE STRENGTH INDICATOR v3 — Hybrid
================================================================================

Base: Estrutura multiplicativa + ATR sizing + hard volume gate (elephant bar)
Adições: Shadow boost convexo + doji handling + gate ergódico + concentração

MUDANÇAS vs ELEPHANT BAR ORIGINAL:
1. shape agora é (body_share + shadow_weight * rejection^α) * close_extreme * (1 - opp_wick)
   → Sombras favoráveis CONTRIBUEM mesmo com corpo pequeno (hammers, dojis)
2. Dojis: direction via close vs midpoint (dragonfly/gravestone não são zerados)
3. Gate ergódico multiplicativo (pós-numba): atenua candles típicos para seu horário
4. Concentração calibrável: P(|CSI|>0.9) ≈ 2% (configurável)

TUDO QUE NÃO ESTÁ LISTADO ACIMA ESTÁ IDÊNTICO AO ORIGINAL.
================================================================================
"""

import numpy as np
from typing import Optional, Dict, Tuple
from datetime import datetime, timezone, timedelta

# Numba fallback: usa njit se disponível, senão no-op decorator
try:
    from numba import njit
except ImportError:
    def njit(*args, **kwargs):
        """Fallback: roda como Python puro se numba não disponível."""
        def decorator(func):
            return func
        if len(args) == 1 and callable(args[0]):
            return args[0]
        return decorator


# ==============================================================================
# HELPERS (idênticos ao original)
# ==============================================================================

@njit(cache=True, nogil=True)
def _clip01(x):
    if x < 0.0:
        return 0.0
    if x > 1.0:
        return 1.0
    return x


@njit(cache=True, nogil=True)
def _ema_fast(arr, period):
    n = arr.shape[0]
    out = np.empty(n, dtype=np.float64)
    if n == 0:
        return out
    p = period
    if p <= 1:
        p = 1
    alpha = 2.0 / (p + 1.0)
    one_minus_alpha = 1.0 - alpha
    out[0] = arr[0]
    for i in range(1, n):
        out[i] = alpha * arr[i] + one_minus_alpha * out[i - 1]
    return out


# ==============================================================================
# CORE NUMBA — ELEPHANT BAR ENHANCED
# ==============================================================================

@njit(cache=True, nogil=True)
def _candle_strength_core(
    open_,
    close_,
    high_,
    low_,
    volume_,
    window_size=1000,
    short_window=20,
    volume_rel_min=1.0,
    # --- NOVOS PARAMS ---
    shadow_alpha=2.0,     # expoente convexo para sombras favoráveis
    shadow_weight=0.40,   # peso do boost de sombra no shape
):
    """
    Core multiplicativo em Numba.
    
    MUDANÇAS vs original:
    
    1) SHAPE com shadow boost:
       ANTES:  shape = body_share * close_extreme * (1 - opp_wick)
       DEPOIS: shape = (body_share + sw * rejection^α) * close_extreme * (1 - opp_wick)
       
       Onde:
         rejection = favorable_shadow_ratio ^ shadow_alpha
         favorable = lower_wick/range (bullish) ou upper_wick/range (bearish)
       
       Por que: um hammer com 88% de sombra inferior e corpo de 8% tinha
       shape ≈ 0.06 no original. Com shadow boost (α=2, sw=0.4):
       shape ≈ 0.06 + 0.4*(0.88²)*0.92*0.95 ≈ 0.33 → captura a rejeição.
    
    2) DOJI handling:
       ANTES:  close == open → direction = 0 → strength = 0 (SEMPRE)
       DEPOIS: close == open → direction via close vs midpoint do range
       
       Dragonfly doji (close=high, long lower shadow):
         direction = +1 (close > midpoint)
         body_share = 0, MAS shadow_boost contribui
         → shape ≈ 0.4 * 1.0 * 1.0 * 1.0 = 0.40 (sinal não-trivial)
       
       Doji simétrico (close ≈ midpoint):
         direction = 0 → strength = 0 (comportamento original preservado)
    """
    n = close_.shape[0]
    strength = np.empty(n, dtype=np.float64)
    shape_out = np.empty(n, dtype=np.float64)
    size_out = np.empty(n, dtype=np.float64)
    volume_gate_out = np.empty(n, dtype=np.float64)
    volume_rel_out = np.empty(n, dtype=np.float64)

    if n == 0:
        return strength, shape_out, size_out, volume_gate_out, volume_rel_out

    long_window = window_size
    if long_window < 20:
        long_window = 20

    short_w = short_window
    if short_w < 2:
        short_w = 2

    volume_window = long_window // 5
    if volume_window < 20:
        volume_window = 20

    # --- True Range + ATR + Volume EMA (idêntico ao original) ---
    tr = np.empty(n, dtype=np.float64)
    tr[0] = high_[0] - low_[0]
    for i in range(1, n):
        a = high_[i] - low_[i]
        b = abs(high_[i] - close_[i - 1])
        c = abs(low_[i] - close_[i - 1])
        m = a
        if b > m:
            m = b
        if c > m:
            m = c
        tr[i] = m

    atr_short = _ema_fast(tr, short_w)
    atr_long = _ema_fast(tr, long_window)
    volume_ema = _ema_fast(volume_, volume_window)

    eps = 1e-12

    for i in range(n):
        o = open_[i]
        c = close_[i]
        h = high_[i]
        l = low_[i]

        body = abs(c - o)
        r = h - l
        if r < eps:
            r = eps

        # --- MUDANÇA 1: Direction com doji handling ---
        if c > o:
            direction = 1.0
        elif c < o:
            direction = -1.0
        else:
            # Doji: usar posição do close vs midpoint do range
            mid = (h + l) * 0.5
            if c > mid + eps:
                direction = 1.0
            elif c < mid - eps:
                direction = -1.0
            else:
                direction = 0.0

        upper = h - max(c, o)
        lower = min(c, o) - l

        body_share = _clip01(body / r)

        if direction > 0.0:
            close_extreme = _clip01((c - l) / r)
            opposite_wick = _clip01(upper / r)
            favorable_wick = _clip01(lower / r)
        elif direction < 0.0:
            close_extreme = _clip01((h - c) / r)
            opposite_wick = _clip01(lower / r)
            favorable_wick = _clip01(upper / r)
        else:
            close_extreme = 0.0
            opposite_wick = 1.0
            favorable_wick = 0.0

        # --- MUDANÇA 2: Shape com shadow boost ---
        # Transformação convexa: sombras grandes contribuem desproporcionalmente
        shadow_rejection = favorable_wick ** shadow_alpha
        shadow_boost = shadow_weight * shadow_rejection

        # Estrutura multiplicativa preservada, mas com termo aditivo no body_share
        # Quando body_share é 0 (doji), shadow_boost ainda contribui
        effective_body = body_share + shadow_boost
        if effective_body > 1.0:
            effective_body = 1.0

        shape = effective_body * close_extreme * (1.0 - opposite_wick)

        # --- SIZE (idêntico ao original) ---
        atr_s = atr_short[i]
        if atr_s < eps:
            atr_s = eps
        atr_l = atr_long[i]
        if atr_l < eps:
            atr_l = eps

        impulse_short = body / atr_s
        impulse_long = body / atr_l
        range_impulse_long = r / atr_l

        body_power = np.sqrt(max(impulse_short, 0.0) * max(impulse_long, 0.0))
        range_power = np.sqrt(max(range_impulse_long, 0.0) * max(impulse_long, 0.0))

        # MUDANÇA MÍNIMA: para dojis, usar range como proxy de impulso
        # Damped por 0.35 para não inflar dojis medianos
        if body < eps and direction != 0.0:
            doji_impulse_s = r / atr_s
            doji_impulse_l = r / atr_l
            body_power = np.sqrt(max(doji_impulse_s, 0.0) * max(doji_impulse_l, 0.0)) * 0.35

        size_raw = 0.95 * body_power + 0.05 * (range_power * body_share)
        size = np.tanh(max(size_raw, 0.0) * 0.55)
        size = _clip01(size)

        # --- VOLUME GATE (idêntico ao original) ---
        vol_base = volume_ema[i]
        if vol_base < eps:
            vol_base = eps
        vol_rel = volume_[i] / vol_base

        if vol_rel < 0.7:
            volume_gate = 0.0
        else:
            z = (vol_rel - volume_rel_min) / 0.12
            logistic = 1.0 / (1.0 + np.exp(-z))
            volume_gate = 0.20 + 0.80 * logistic

        # --- FINAL (idêntico ao original) ---
        core_strength = 0.68 * size + 0.32 * shape
        magnitude = core_strength * volume_gate
        if magnitude > 1.0:
            magnitude = 1.0

        strength[i] = direction * magnitude
        shape_out[i] = shape
        size_out[i] = size
        volume_gate_out[i] = volume_gate
        volume_rel_out[i] = vol_rel

    return strength, shape_out, size_out, volume_gate_out, volume_rel_out


# ==============================================================================
# GATE ERGÓDICO (numpy — pós-numba)
# ==============================================================================

def _parse_timestamps(timestamps):
    """Converte qualquer formato para datetime objects."""
    if len(timestamps) == 0:
        return np.array([], dtype=object)
    s = timestamps[0]
    if isinstance(s, datetime):
        return np.asarray(timestamps, dtype=object)
    if isinstance(s, np.datetime64):
        return np.array([
            datetime.fromtimestamp(
                (t - np.datetime64('1970-01-01T00:00:00')) / np.timedelta64(1, 's'),
                tz=timezone.utc)
            for t in timestamps], dtype=object)
    if isinstance(s, (int, float, np.integer, np.floating)):
        return np.array([datetime.fromtimestamp(float(t), tz=timezone.utc)
                         for t in timestamps], dtype=object)
    if isinstance(s, str):
        return np.array([datetime.fromisoformat(t.replace('Z', '+00:00'))
                         for t in timestamps], dtype=object)
    raise ValueError(f"Tipo de timestamp não reconhecido: {type(s)}")


def _detect_timeframe(timestamps):
    """Auto-detecta timeframe pela mediana dos deltas."""
    dt = _parse_timestamps(timestamps)
    if len(dt) < 2:
        return {'median_seconds': 86400, 'label': 'daily', 'category': 'daily'}
    deltas = [(dt[i] - dt[i-1]).total_seconds()
              for i in range(1, len(dt))
              if (dt[i] - dt[i-1]).total_seconds() > 0]
    if not deltas:
        return {'median_seconds': 86400, 'label': 'daily', 'category': 'daily'}
    md = np.median(deltas)
    if md < 60:     return {'median_seconds': md, 'label': f'{int(md)}s', 'category': 'sub_minute'}
    elif md < 300:  return {'median_seconds': md, 'label': f'{int(md/60)}min', 'category': 'minute'}
    elif md < 900:  return {'median_seconds': md, 'label': f'{int(md/60)}min', 'category': 'multi_minute'}
    elif md < 3600: return {'median_seconds': md, 'label': f'{int(md/60)}min', 'category': 'sub_hour'}
    elif md < 14400:return {'median_seconds': md, 'label': f'{int(md/3600)}H', 'category': 'hourly'}
    elif md < 604800:return {'median_seconds': md, 'label': 'daily', 'category': 'daily'}
    elif md < 2592000:return {'median_seconds': md, 'label': 'weekly', 'category': 'weekly'}
    else:           return {'median_seconds': md, 'label': 'monthly', 'category': 'monthly'}


def _create_temporal_buckets(timestamps, tf_info):
    """Cria buckets fine + coarse para ergodic gate."""
    dt = _parse_timestamps(timestamps)
    n = len(dt)
    cat = tf_info['category']
    fine = np.zeros(n, dtype=np.int32)
    coarse = np.zeros(n, dtype=np.int32)
    for i, d in enumerate(dt):
        h, m, dow = d.hour, d.minute, d.weekday()
        if cat == 'sub_minute':
            fine[i], coarse[i] = h*60+m, h
        elif cat == 'minute':
            fine[i], coarse[i] = (h*60+m)//5, h
        elif cat == 'multi_minute':
            s15 = (h*60+m)//15
            fine[i], coarse[i] = s15*7+dow, s15
        elif cat == 'sub_hour':
            fine[i], coarse[i] = h*7+dow, h
        elif cat == 'hourly':
            sess = 0 if h < 8 else (1 if h < 14 else 2)
            fine[i], coarse[i] = sess*7+dow, sess
        elif cat == 'daily':
            fine[i] = coarse[i] = dow
        elif cat == 'weekly':
            fine[i] = coarse[i] = min((d.day-1)//7, 4)
        else:
            fine[i] = coarse[i] = d.month - 1
    return fine, coarse


def _compute_ergodic_gate(
    values,
    fine_buckets,
    coarse_buckets,
    min_samples_fine=20,
    min_samples_coarse=10,
    lambda_max=0.35,
    eta=0.8,
):
    """
    Gate ergódico multiplicativo.
    
    unusualness = |2 * rank - 1|   (0 = típico, 1 = excepcional)
    gate = 1 - λ * conf * (1 - unusualness^η)
    
    Gate ∈ (1-λ, 1.0] — NUNCA infla, apenas atenua ou confirma.
    Causalidade estrita: só usa dados passados.
    """
    n = len(values)
    ergodic_ranks = np.full(n, 0.5)
    confidence = np.zeros(n)
    gate = np.ones(n)

    fine_hist = {}
    coarse_hist = {}

    for i in range(n):
        fb, cb = int(fine_buckets[i]), int(coarse_buckets[i])
        val = values[i]
        rank, conf = 0.5, 0.0

        if fb in fine_hist and len(fine_hist[fb]) >= min_samples_fine:
            past = np.array(fine_hist[fb])
            rank = np.sum(past <= val) / len(past)
            conf = 1.0
        elif cb in coarse_hist and len(coarse_hist[cb]) >= min_samples_coarse:
            past = np.array(coarse_hist[cb])
            rank = np.sum(past <= val) / len(past)
            conf = 0.7

        ergodic_ranks[i] = rank
        confidence[i] = conf

        unusualness = abs(2.0 * rank - 1.0)
        gate[i] = 1.0 - lambda_max * conf * (1.0 - unusualness ** eta)

        fine_hist.setdefault(fb, []).append(val)
        coarse_hist.setdefault(cb, []).append(val)

    return gate, ergodic_ranks, confidence


# ==============================================================================
# CONCENTRAÇÃO CALIBRADA
# ==============================================================================

def _concentration_exponent(threshold=0.9, tail_pct=0.02):
    """
    p = ln(threshold) / ln(1 - tail_pct)
    
    threshold=0.9, tail_pct=0.02 → p=5.22 → ~2% com |CSI|>0.9
    """
    return np.log(threshold) / np.log(1.0 - tail_pct)


def _concentrate(values, p):
    """sign(x) * |x|^p — comprime distribuição ao redor de zero."""
    return np.sign(values) * np.power(np.abs(np.clip(values, -1, 1)), p)


# ==============================================================================
# FUNÇÃO PRINCIPAL
# ==============================================================================

def candle_strength_indicator(
    open_: np.ndarray,
    close_: np.ndarray,
    high_: np.ndarray,
    low_: np.ndarray,
    volume_: np.ndarray,
    timestamps: Optional[np.ndarray] = None,
    # --- Params do core (elephant bar original) ---
    window_size: int = 1000,
    short_window: int = 20,
    volume_rel_min: float = 1.0,
    # --- Params novos (shadow boost) ---
    shadow_alpha: float = 2.0,
    shadow_weight: float = 0.40,
    # --- Gate ergódico ---
    ergodic_lambda: float = 0.35,
    ergodic_eta: float = 0.8,
    min_samples_fine: int = 20,
    min_samples_coarse: int = 10,
    # --- Concentração ---
    concentration_threshold: float = 0.9,
    concentration_tail_pct: float = 0.02,
    use_concentration: bool = False,
    # --- Output ---
    return_components: bool = False,
) -> Dict[str, np.ndarray]:
    """
    Candle Strength Indicator v3 — Hybrid.
    
    Pipeline:
    1. Core Numba (elephant bar enhanced) → strength bruto ∈ [-1, +1]
    2. Concentração (opcional) → comprime distribuição
    3. Gate ergódico (se timestamps) → atenua candles típicos p/ seu horário
    
    Params:
    -------
    open_, close_, high_, low_, volume_: arrays OHLCV
    timestamps: array de timestamps (qualquer formato). None = sem ergódico.
    window_size: janela longa para ATR (default 1000)
    short_window: janela curta para ATR (default 20)
    volume_rel_min: threshold de volume relativo para logistic gate
    shadow_alpha: expoente convexo para sombras favoráveis (default 2.0)
    shadow_weight: peso do shadow boost no shape (default 0.40)
    ergodic_lambda: atenuação máxima do gate ergódico (default 0.35)
    ergodic_eta: generosidade do gate (default 0.8)
    min_samples_fine/coarse: mínimos para buckets ergódicos
    concentration_threshold: limiar para calibração (default 0.9)
    concentration_tail_pct: fração acima do threshold (default 0.02 = 2%)
    use_concentration: True/False para ativar concentração
    return_components: se True, retorna intermediários
    
    Retorna:
    --------
    dict com 'csi' ∈ [-1, +1] e opcionalmente componentes.
    """
    # Garantir contiguidade (Numba exige)
    o = np.ascontiguousarray(open_, dtype=np.float64)
    c = np.ascontiguousarray(close_, dtype=np.float64)
    h = np.ascontiguousarray(high_, dtype=np.float64)
    l = np.ascontiguousarray(low_, dtype=np.float64)
    v = np.ascontiguousarray(volume_, dtype=np.float64)

    # === Estágio 1: Core Numba ===
    strength, shape, size, vol_gate, vol_rel = _candle_strength_core(
        o, c, h, l, v,
        window_size=window_size,
        short_window=short_window,
        volume_rel_min=volume_rel_min,
        shadow_alpha=shadow_alpha,
        shadow_weight=shadow_weight,
    )

    csi = strength.copy()

    # === Estágio 2: Concentração (opcional) ===
    conc_p = None
    if use_concentration:
        conc_p = _concentration_exponent(concentration_threshold,
                                          concentration_tail_pct)
        csi = _concentrate(csi, conc_p)

    # === Estágio 3: Gate ergódico (se timestamps fornecidos) ===
    has_ergodic = timestamps is not None and len(timestamps) > 0
    ergodic_gate = np.ones(len(csi))
    ergodic_rank = np.full(len(csi), 0.5)
    ergodic_conf = np.zeros(len(csi))
    tf_info = None

    if has_ergodic:
        tf_info = _detect_timeframe(timestamps)
        fine_b, coarse_b = _create_temporal_buckets(timestamps, tf_info)
        ergodic_gate, ergodic_rank, ergodic_conf = _compute_ergodic_gate(
            np.abs(strength),  # Gate opera sobre magnitude absoluta
            fine_b, coarse_b,
            min_samples_fine=min_samples_fine,
            min_samples_coarse=min_samples_coarse,
            lambda_max=ergodic_lambda,
            eta=ergodic_eta,
        )
        csi = csi * ergodic_gate

    csi = np.clip(csi, -1.0, 1.0)

    result = {'csi': csi}

    if return_components:
        result.update({
            'strength_raw': strength,
            'shape': shape,
            'size': size,
            'volume_gate': vol_gate,
            'volume_rel': vol_rel,
            'ergodic_gate': ergodic_gate,
            'ergodic_rank': ergodic_rank,
            'ergodic_confidence': ergodic_conf,
            'concentration_p': conc_p,
            'timeframe_info': tf_info,
        })

    return result


# ==============================================================================
# EXEMPLOS
# ==============================================================================

def run_examples():
    print("=" * 76)
    print("CSI v3 HYBRID — VALIDAÇÃO")
    print("=" * 76)

    # =================================================================
    # PARTE 1: SHADOW BOOST — IMPACTO EM CANDLES CLÁSSICOS
    # =================================================================
    print("\n" + "-" * 76)
    print("PARTE 1: SHADOW BOOST — ORIGINAL vs ENHANCED")
    print("-" * 76 + "\n")

    # Simular uma série mínima para ATR warm-up + candle de teste
    np.random.seed(42)
    n_warmup = 100
    base_price = 100.0

    # Warmup: candles normais com range ~1.0
    wo = np.full(n_warmup, base_price)
    wc = wo + np.random.normal(0, 0.3, n_warmup)
    wh = np.maximum(wo, wc) + np.abs(np.random.normal(0, 0.2, n_warmup))
    wl = np.minimum(wo, wc) - np.abs(np.random.normal(0, 0.2, n_warmup))
    wv = np.full(n_warmup, 1000.0)

    test_candles = [
        ("Marubozu Bullish",  100.0, 110.0, 110.0, 100.0, 1500.0),
        ("Marubozu Bearish",  110.0, 100.0, 110.0, 100.0, 1500.0),
        ("Hammer clássico",   108.0, 108.5, 109.0, 100.0, 1500.0),
        ("Shooting Star",     102.0, 101.5, 110.0, 101.0, 1500.0),
        ("Dragonfly Doji",    110.0, 110.0, 110.0, 100.0, 1500.0),
        ("Gravestone Doji",   100.0, 100.0, 110.0, 100.0, 1500.0),
        ("Doji simétrico",    105.0, 105.0, 108.0, 102.0, 1500.0),
        ("Hammer vol baixo",  108.0, 108.5, 109.0, 100.0, 300.0),
    ]

    print(f"  {'Candle':<22} {'Original':>10} {'Enhanced':>10} {'Δ':>8}  Notas")
    print("  " + "-" * 70)

    for name, to, tc, th, tl, tv in test_candles:
        # Append candle de teste
        o = np.append(wo, to)
        c = np.append(wc, tc)
        h = np.append(wh, th)
        l = np.append(wl, tl)
        v = np.append(wv, tv)

        # Original (sem shadow boost)
        s_orig, _, _, _, _ = _candle_strength_core(
            o, c, h, l, v,
            window_size=100, short_window=20, volume_rel_min=1.0,
            shadow_alpha=2.0, shadow_weight=0.0,  # ← DESLIGADO
        )
        val_orig = s_orig[-1]

        # Enhanced (com shadow boost)
        s_enh, shape_e, size_e, vg_e, _ = _candle_strength_core(
            o, c, h, l, v,
            window_size=100, short_window=20, volume_rel_min=1.0,
            shadow_alpha=2.0, shadow_weight=0.40,  # ← LIGADO
        )
        val_enh = s_enh[-1]

        delta = val_enh - val_orig
        notes = ""
        if abs(delta) < 0.001:
            notes = "(sem mudança — não tem sombra favorável)"
        elif "Doji" in name and abs(val_orig) < 0.001:
            notes = f"(doji ativado! shape={shape_e[-1]:.3f})"
        elif abs(delta) > 0.05:
            notes = f"(shadow boost +{delta:+.3f})"

        print(f"  {name:<22} {val_orig:>+10.4f} {val_enh:>+10.4f} {delta:>+8.4f}  {notes}")

    # =================================================================
    # PARTE 2: DISTRIBUIÇÃO NATURAL DO ELEPHANT BAR
    # =================================================================
    print("-" * 76)
    print("PARTE 2: DISTRIBUIÇÃO — JÁ CONCENTRADA NATURALMENTE")
    print("-" * 76)

    # Gerar série longa para análise de distribuição
    np.random.seed(42)
    n = 5000
    price = 100.0
    o_all, c_all, h_all, l_all, v_all, ts_all = [], [], [], [], [], []
    base_dt = datetime(2024, 1, 2, tzinfo=timezone.utc)
    cpd = 78

    for i in range(n):
        day = i // cpd
        ci = i % cpd
        mins = ci * 5
        hr = 9 + (30 + mins) // 60
        mn = (30 + mins) % 60
        day_dt = base_dt + timedelta(days=day)
        ts_all.append(day_dt.replace(hour=hr, minute=mn))

        progress = ci / cpd
        vol_intra = 2.0 * (progress - 0.5) ** 2 + 0.3

        body = np.random.normal(0.02, vol_intra * 0.6)
        oi = price
        ci_v = oi + body
        hi = max(oi, ci_v) + abs(np.random.exponential(vol_intra * 0.3))
        li = min(oi, ci_v) - abs(np.random.exponential(vol_intra * 0.3))
        vi = max(100, 1000 * vol_intra + np.random.normal(0, 100))

        o_all.append(oi); c_all.append(ci_v)
        h_all.append(hi); l_all.append(li); v_all.append(vi)
        price = ci_v

    o_a = np.array(o_all); c_a = np.array(c_all)
    h_a = np.array(h_all); l_a = np.array(l_all)
    v_a = np.array(v_all); ts_a = np.array(ts_all)

    # Default (sem concentração extra)
    res_default = candle_strength_indicator(
        o_a, c_a, h_a, l_a, v_a,
        return_components=True)

    # Com concentração (para referência)
    res_conc = candle_strength_indicator(
        o_a, c_a, h_a, l_a, v_a,
        use_concentration=True, return_components=True)

    # Com ergódico
    res_full = candle_strength_indicator(
        o_a, c_a, h_a, l_a, v_a, timestamps=ts_a,
        return_components=True)

    print(f"""
  A estrutura multiplicativa + tanh do elephant bar já produz uma 
  distribuição naturalmente concentrada (diferente do quantile rank 
  do CSI puro, que é uniforme). Concentração extra é OPCIONAL.
    """)

    conc_p_val = res_conc['concentration_p']
    print(f"  {'Threshold':>12} {'Default':>10} {'+ Ergódico':>11} {f'+ Conc(p={conc_p_val:.1f})':>14}")
    print("  " + "-" * 50)

    for t in [0.1, 0.2, 0.3, 0.5, 0.7, 0.8, 0.9]:
        pd_ = np.mean(np.abs(res_default['csi']) > t) * 100
        pf = np.mean(np.abs(res_full['csi']) > t) * 100
        pc = np.mean(np.abs(res_conc['csi']) > t) * 100
        print(f"  |CSI|>{t:.2f}:  {pd_:>9.2f}% {pf:>10.2f}% {pc:>13.2f}%")

    print("""
  Default (sem concentração): distribuição já é conservadora.
  Valores > 0.8 são naturalmente raros (~1-3%).
  
  Concentração extra: disponível via use_concentration=True se necessário,
  mas com a estrutura do elephant bar normalmente NÃO é preciso.
  
  Gate ergódico: atenua candles típicos para seu horário, reduz falsos positivos.
    """)

    # =================================================================
    # PARTE 3: GATE ERGÓDICO
    # =================================================================
    print("-" * 76)
    print("PARTE 3: GATE ERGÓDICO")
    print("-" * 76)

    tf = res_full['timeframe_info']
    print(f"\n  Timeframe: {tf['label']} ({tf['category']})")

    tail = min(20 * cpd, n)
    start = n - tail
    hours = np.array([ts_all[i].hour + ts_all[i].minute / 60 for i in range(n)])
    periods = [
        ("Abertura (9:30-10:30)",  9.5, 10.5),
        ("Almoço  (12:00-13:00)", 12.0, 13.0),
        ("Fech.   (15:00-16:00)", 15.0, 16.0),
    ]

    print(f"\n  {'Período':<28} {'|CSI| sem':>10} {'|CSI| +erg':>11} "
          f"{'Gate méd':>9} {'Conf':>6}")
    print("  " + "-" * 66)

    for pname, t0, t1 in periods:
        mask = (hours[start:] >= t0) & (hours[start:] < t1)
        idx = np.where(mask)[0] + start
        if len(idx) == 0:
            continue
        csn = np.mean(np.abs(res_default['csi'][idx]))   # sem ergódico
        csf = np.mean(np.abs(res_full['csi'][idx]))       # com ergódico
        gm = np.mean(res_full['ergodic_gate'][idx])
        co = np.mean(res_full['ergodic_confidence'][idx])
        print(f"  {pname:<28} {csn:>10.4f} {csf:>11.4f} {gm:>9.4f} {co:>6.2f}")

    print("""
  Gate < 1.0: candles típicos para seu horário são atenuados.
  |CSI| com gate ≤ sem gate: NUNCA infla.
    """)

    # =================================================================
    # PARTE 4: DEGRADAÇÃO GRACEFUL
    # =================================================================
    print("-" * 76)
    print("PARTE 4: DEGRADAÇÃO GRACEFUL")
    print("-" * 76 + "\n")

    for nc in [10, 50, 200, 1000, n]:
        r = candle_strength_indicator(
            o_a[:nc], c_a[:nc], h_a[:nc], l_a[:nc], v_a[:nc],
            timestamps=ts_a[:nc], return_components=True)
        ac = np.mean(r['ergodic_confidence'])
        ag = np.mean(r['ergodic_gate'])
        pa = np.mean(r['ergodic_confidence'] > 0) * 100
        print(f"  {nc:>6} candles → Conf: {ac:.3f}  "
              f"Gate: {ag:.4f}  Buckets: {pa:.0f}%")

    print("""
  Poucos dados → gate=1.0 → elephant bar puro (original)
  Dados abundantes → gate ativo → proteção extra
    """)

    print("=" * 76)
    print("FIM — CSI v3 Hybrid")
    print("=" * 76)


if __name__ == "__main__":
    run_examples()

#### Transformação dos dados

In [ ]:
cs = candle_strength_indicator(
        df.open.values, df.high.values, df.low.values, df.close.values, df.volume.values, df.index.values, return_components=True,
        short_window=1000
    )
cs
    

#### Visualização gráfica

In [ ]:
df['csi'] = cs['csi']

In [ ]:
import numpy as np
from numba import njit

@njit(nogil=True, cache=True)
def _bisect_left(a, n, x):
    lo = 0
    hi = n
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] < x:
            lo = mid + 1
        else:
            hi = mid
    return lo


@njit(nogil=True, cache=True)
def _quantile_from_sorted_prefix(sorted_vals, n, q):
    if n == 0:
        return np.nan

    if q <= 0.0:
        return sorted_vals[0]
    if q >= 1.0:
        return sorted_vals[n - 1]

    idx = q * (n - 1)
    lo = int(np.floor(idx))
    hi = int(np.ceil(idx))

    if lo == hi:
        return sorted_vals[lo]

    frac = idx - lo
    return sorted_vals[lo] + (sorted_vals[hi] - sorted_vals[lo]) * frac


@njit(nogil=True, cache=True)
def rolling_quantile_incremental(arr, window, q):
    """
    Rolling quantile exato, causal e incremental.
    
    Regras:
    - out[i] usa somente arr[max(0, i-window+1):i+1]
    - sem leitura para frente
    - se houver NaN na janela completa, out[i] = NaN
    - mesmo comportamento do seu código original para janela incompleta:
      índices < window-1 ficam como NaN
    """
    n = arr.shape[0]
    out = np.empty(n, dtype=np.float64)
    out[:] = np.nan

    if window <= 0:
        return out

    # Janela ordenada contendo apenas os valores válidos da janela atual
    sorted_vals = np.empty(window, dtype=np.float64)

    # Ring buffer com os valores brutos da janela para saber quem sai
    ring = np.empty(window, dtype=np.float64)
    ring[:] = np.nan

    # Quantidade de valores válidos atualmente na janela
    m = 0

    for i in range(n):
        slot = i % window

        # 1) Remove o valor que está saindo da janela
        if i >= window:
            old_v = ring[slot]

            if not np.isnan(old_v):
                pos = _bisect_left(sorted_vals, m, old_v)

                # Remove a primeira ocorrência encontrada
                # shift à esquerda
                for j in range(pos, m - 1):
                    sorted_vals[j] = sorted_vals[j + 1]

                m -= 1

        # 2) Insere o valor novo
        new_v = arr[i]
        ring[slot] = new_v

        if not np.isnan(new_v):
            pos = _bisect_left(sorted_vals, m, new_v)

            # shift à direita para abrir espaço
            for j in range(m, pos, -1):
                sorted_vals[j] = sorted_vals[j - 1]

            sorted_vals[pos] = new_v
            m += 1

        # 3) Só calcula quando a janela está completa
        #    e sem NaN (m == window)
        if i >= window - 1 and m == window:
            out[i] = _quantile_from_sorted_prefix(sorted_vals, m, q)

    return out

In [ ]:
"""
Candle Strength Indicator v2 — Refatoração com melhorias estruturais
====================================================================

Changelog vs v1:
  1. Shape: média ponderada em vez de produto triplo (menos punitivo)
  2. Size: range_power com peso real (15%), fator tanh configurável
  3. Volume Gate: transição suave (sem descontinuidade), centro configurável
  4. Direction: contínuo (não ternário), incorpora posição do close na range
  5. Composição: termo de interação shape×size, penaliza pavios em velas grandes
  6. Contexto sequencial: streak e momentum de N barras
  7. Parâmetros expostos e documentados
"""

import numpy as np
import pandas as pd
from numba import njit


# ============================================================
# Helpers (mantidos da v1, sem alteração)
# ============================================================

def _require_volume_column(data: pd.DataFrame, strategy_name: str) -> np.ndarray:
    if "volume" not in data.columns:
        raise ValueError(f"{strategy_name} requires a 'volume' column.")
    return np.ascontiguousarray(data["volume"].to_numpy(dtype=np.float64, copy=False))


@njit(cache=True, nogil=True)
def _clip01(x):
    if x < 0.0:
        return 0.0
    if x > 1.0:
        return 1.0
    return x


@njit(cache=True, nogil=True)
def _ema_fast(arr, period):
    n = arr.shape[0]
    out = np.empty(n, dtype=np.float64)
    if n == 0:
        return out
    p = max(period, 1)
    alpha = 2.0 / (p + 1.0)
    one_minus_alpha = 1.0 - alpha
    out[0] = arr[0]
    for i in range(1, n):
        out[i] = alpha * arr[i] + one_minus_alpha * out[i - 1]
    return out


# ============================================================
# MELHORIA 1: Shape com média ponderada (não produto triplo)
# ============================================================
# 
# Problema v1: shape = body_share × close_extreme × (1 - opposite_wick)
#   - Qualquer fator em 0.5 colapsa o resultado para ~0.125
#   - Velas "boas mas não perfeitas" recebem scores muito baixos
#
# Solução v2: média ponderada com pesos configuráveis + bonus multiplicativo
#   - Componente aditivo garante score razoável para velas boas
#   - Bonus multiplicativo premia velas excepcionais
#
@njit(cache=True, nogil=True)
def _shape_v2(body_share, close_extreme, opposite_wick,
              w_body=0.45, w_close=0.30, w_wick=0.25,
              interaction_bonus=0.15):
    """
    Retorna shape em [0, 1].
    
    v1: body_share * close_extreme * (1 - opp_wick)         → muito punitivo
    v2: weighted_avg + bonus * (body_share * close_extreme)  → mais balanceado
    """
    clean_wick = 1.0 - opposite_wick
    
    # Componente aditivo (base)
    base = w_body * body_share + w_close * close_extreme + w_wick * clean_wick
    
    # Bonus para velas excepcionais (todas as dimensões altas)
    interaction = interaction_bonus * body_share * close_extreme * clean_wick
    
    result = base + interaction
    return _clip01(result)


# ============================================================
# MELHORIA 2: Direction contínuo (não ternário)
# ============================================================
#
# Problema v1: direction ∈ {-1, 0, +1}
#   - Perde informação: close $0.01 acima do open = +1, igual a close no high
#   - Doji sempre = 0, mas doji após tendência forte é sinal importante
#
# Solução v2: direction contínuo baseado na posição do close na range
#
@njit(cache=True, nogil=True)
def _direction_continuous(close, open_, high, low, eps=1e-12):
    """
    Retorna valor em [-1, +1] baseado na posição do close na range.
    
    close == high → +1.0
    close == low  → -1.0
    close == mid  →  0.0
    
    Mais informativo que o ternário da v1.
    """
    r = high - low
    if r < eps:
        return 0.0
    # Posição normalizada: 0 = low, 1 = high
    pos = (close - low) / r
    # Mapeia [0, 1] → [-1, +1]
    return 2.0 * pos - 1.0


# ============================================================
# MELHORIA 3: Volume Gate sem descontinuidade
# ============================================================
#
# Problema v1: 
#   - Cutoff abrupto em 0.7 (0→20% instantâneo)
#   - Centro da logística em vol_rel=1.0 (volume médio penalizado)
#   - Janela de volume muito longa
#
# Solução v2: logística pura sem cutoff, centro mais baixo
#
@njit(cache=True, nogil=True)
def _volume_gate_v2(vol_rel, center=0.85, steepness=8.0, floor=0.05):
    """
    Gate suave via logística pura.
    
    center:    vol_rel onde gate ≈ 0.5 (v1 usava 1.0, muito alto)
    steepness: quão rápida é a transição (v1 usava 1/0.12 ≈ 8.3)
    floor:     gate mínimo (v1 tinha 0 abaixo de 0.7, depois 0.2)
    
    Sem descontinuidade: transição é C∞ (infinitamente suave).
    """
    z = steepness * (vol_rel - center)
    logistic = 1.0 / (1.0 + np.exp(-z))
    return floor + (1.0 - floor) * logistic


# ============================================================
# MELHORIA 4: Composição com interação shape×size
# ============================================================
#
# Problema v1: core_strength = 0.68*size + 0.32*shape (linear)
#   - size=0.8 + shape=0.1 → 0.576 (alto, mas vela tem pavios enormes!)
#   - Não penaliza a contradição "corpo grande + forma ruim"
#
# Solução v2: adiciona termo de interação
#
@njit(cache=True, nogil=True)
def _compose_strength_v2(size, shape, volume_gate, direction,
                         w_size=0.55, w_shape=0.25, w_interact=0.20):
    """
    core = w_size*size + w_shape*shape + w_interact*(size*shape)
    
    O termo size*shape garante que velas com score alto em ambos
    são premiadas, e velas contraditórias são penalizadas.
    
    Exemplo:
      size=0.8, shape=0.8 → 0.55*0.8 + 0.25*0.8 + 0.20*0.64 = 0.768
      size=0.8, shape=0.1 → 0.55*0.8 + 0.25*0.1 + 0.20*0.08 = 0.481
      
    vs v1:
      size=0.8, shape=0.1 → 0.68*0.8 + 0.32*0.1 = 0.576  (inflado!)
    """
    core = w_size * size + w_shape * shape + w_interact * (size * shape)
    magnitude = _clip01(core * volume_gate)
    return direction * magnitude


# ============================================================
# MELHORIA 5: Contexto sequencial (streak / momentum)
# ============================================================
#
# Problema v1: cada vela é avaliada 100% isoladamente
#   - Não diferencia "primeira vela forte após consolidação" de 
#     "quinta vela forte consecutiva" (exaustão)
#   - Padrões multi-candle invisíveis
#
# Solução v2: calcular streak e momentum como outputs adicionais
#
@njit(cache=True, nogil=True)
def _sequential_context(strength, lookback=5):
    """
    Calcula contexto sequencial para cada barra.
    
    Retorna:
      streak:    nº de barras consecutivas na mesma direção (com sinal)
      momentum:  média da strength nas últimas `lookback` barras
      freshness: 1.0 se é a 1ª vela forte, decai com repetição
                 (captura "breakout" vs "exaustão")
    """
    n = strength.shape[0]
    streak = np.zeros(n, dtype=np.float64)
    momentum = np.zeros(n, dtype=np.float64)
    freshness = np.ones(n, dtype=np.float64)
    
    if n == 0:
        return streak, momentum, freshness
    
    # Streak
    current_streak = 0.0
    for i in range(n):
        s = strength[i]
        if s > 0.0:
            if current_streak > 0:
                current_streak += 1.0
            else:
                current_streak = 1.0
        elif s < 0.0:
            if current_streak < 0:
                current_streak -= 1.0
            else:
                current_streak = -1.0
        else:
            current_streak = 0.0
        streak[i] = current_streak
    
    # Momentum (média móvel simples do strength)
    cumsum = 0.0
    for i in range(n):
        cumsum += strength[i]
        if i >= lookback:
            cumsum -= strength[i - lookback]
            momentum[i] = cumsum / lookback
        else:
            momentum[i] = cumsum / (i + 1)
    
    # Freshness: decai exponencialmente com streak
    # streak=1 → 1.0, streak=3 → ~0.7, streak=6 → ~0.4
    decay_rate = 0.15
    for i in range(n):
        abs_streak = abs(streak[i])
        if abs_streak <= 1.0:
            freshness[i] = 1.0
        else:
            freshness[i] = np.exp(-decay_rate * (abs_streak - 1.0))
    
    return streak, momentum, freshness


# ============================================================
# FUNÇÃO PRINCIPAL v2
# ============================================================

@njit(cache=True, nogil=True)
def _candle_strength_v2(
    open_,
    close_,
    high_,
    low_,
    volume_,
    # --- Parâmetros de ATR ---
    window_size=1000,
    short_window=20,
    # --- Parâmetros de Shape (NOVO: configuráveis) ---
    shape_w_body=0.45,
    shape_w_close=0.30,
    shape_w_wick=0.25,
    shape_interaction=0.15,
    # --- Parâmetros de Size ---
    tanh_factor=0.55,
    range_power_weight=0.15,       # v1: 0.05 (irrelevante) → v2: 0.15
    # --- Parâmetros de Volume Gate (NOVO: configuráveis) ---
    vol_center=0.85,               # v1: 1.0 → v2: 0.85
    vol_steepness=8.0,
    vol_floor=0.05,                # v1: 0.0/0.20 (descontínuo) → v2: 0.05
    vol_window_divisor=5,
    # --- Parâmetros de Composição (NOVO: com interação) ---
    comp_w_size=0.55,              # v1: 0.68
    comp_w_shape=0.25,             # v1: 0.32
    comp_w_interact=0.20,          # v1: 0.00 (não existia)
    # --- Parâmetros de Contexto ---
    context_lookback=5,
):
    n = close_.shape[0]
    
    # Outputs principais
    strength = np.empty(n, dtype=np.float64)
    shape_out = np.empty(n, dtype=np.float64)
    size_out = np.empty(n, dtype=np.float64)
    volume_gate_out = np.empty(n, dtype=np.float64)
    volume_rel_out = np.empty(n, dtype=np.float64)
    direction_out = np.empty(n, dtype=np.float64)
    
    if n == 0:
        empty = np.empty(0, dtype=np.float64)
        return (strength, shape_out, size_out, volume_gate_out, 
                volume_rel_out, direction_out, empty, empty, empty)
    
    # --- ATR (sem alteração) ---
    long_window = max(window_size, 20)
    short_w = max(short_window, 2)
    volume_window = max(long_window // vol_window_divisor, 20)
    
    tr = np.empty(n, dtype=np.float64)
    tr[0] = high_[0] - low_[0]
    for i in range(1, n):
        a = high_[i] - low_[i]
        b = abs(high_[i] - close_[i - 1])
        c = abs(low_[i] - close_[i - 1])
        m = a
        if b > m:
            m = b
        if c > m:
            m = c
        tr[i] = m
    
    atr_short = _ema_fast(tr, short_w)
    atr_long = _ema_fast(tr, long_window)
    volume_ema = _ema_fast(volume_, volume_window)
    
    eps = 1e-12
    body_pw = 1.0 - range_power_weight   # v2: 0.85 (v1: 0.95)
    
    for i in range(n):
        o = open_[i]
        c = close_[i]
        h = high_[i]
        l = low_[i]
        
        body = abs(c - o)
        r = h - l
        if r < eps:
            r = eps
        
        # --- MELHORIA 2: Direction contínuo ---
        direction = _direction_continuous(c, o, h, l, eps)
        direction_out[i] = direction
        
        # --- Componentes de shape ---
        upper = h - max(c, o)
        lower = min(c, o) - l
        
        body_share = _clip01(body / r)
        
        if c >= o:  # bullish ou doji
            close_extreme = _clip01((c - l) / r)
            opposite_wick = _clip01(upper / r)
        else:       # bearish
            close_extreme = _clip01((h - c) / r)
            opposite_wick = _clip01(lower / r)
        
        # --- MELHORIA 1: Shape v2 ---
        shape = _shape_v2(body_share, close_extreme, opposite_wick,
                          shape_w_body, shape_w_close, shape_w_wick,
                          shape_interaction)
        shape_out[i] = shape
        
        # --- Size (melhorado: range_power com peso real) ---
        atr_s = max(atr_short[i], eps)
        atr_l = max(atr_long[i], eps)
        
        impulse_short = body / atr_s
        impulse_long = body / atr_l
        range_impulse_long = r / atr_l
        
        body_power = np.sqrt(max(impulse_short * impulse_long, 0.0))
        range_power = np.sqrt(max(range_impulse_long * impulse_long, 0.0))
        size_raw = body_pw * body_power + range_power_weight * (range_power * body_share)
        size = _clip01(np.tanh(max(size_raw, 0.0) * tanh_factor))
        size_out[i] = size
        
        # --- MELHORIA 3: Volume gate suave ---
        vol_base = max(volume_ema[i], eps)
        vol_rel = volume_[i] / vol_base
        volume_rel_out[i] = vol_rel
        
        vgate = _volume_gate_v2(vol_rel, vol_center, vol_steepness, vol_floor)
        volume_gate_out[i] = vgate
        
        # --- MELHORIA 4: Composição com interação ---
        strength[i] = _compose_strength_v2(
            size, shape, vgate, direction,
            comp_w_size, comp_w_shape, comp_w_interact
        )
    
    # --- MELHORIA 5: Contexto sequencial ---
    streak, momentum, freshness = _sequential_context(strength, context_lookback)
    
    return (strength, shape_out, size_out, volume_gate_out,
            volume_rel_out, direction_out, streak, momentum, freshness)


# ============================================================
# WRAPPER PANDAS (para uso prático)
# ============================================================

def candle_strength(data: pd.DataFrame, **kwargs) -> pd.DataFrame:
    """
    Calcula o indicador de força de candle v2.
    
    Parameters
    ----------
    data : DataFrame com colunas: open, close, high, low, volume
    **kwargs : parâmetros opcionais passados para _candle_strength_v2
    
    Returns
    -------
    DataFrame com colunas:
        strength      : score principal [-1, +1]
        shape         : qualidade da forma [0, 1]
        size          : tamanho relativo [0, 1]
        volume_gate   : confirmação de volume [0, 1]
        volume_rel    : volume relativo ao EMA
        direction     : direção contínua [-1, +1]
        streak        : barras consecutivas na mesma direção
        momentum      : média de strength recente
        freshness     : decaimento por repetição [0, 1]
    """
    vol = _require_volume_column(data, "candle_strength")
    
    result = _candle_strength_v2(
        np.ascontiguousarray(data["open"].to_numpy(dtype=np.float64)),
        np.ascontiguousarray(data["close"].to_numpy(dtype=np.float64)),
        np.ascontiguousarray(data["high"].to_numpy(dtype=np.float64)),
        np.ascontiguousarray(data["low"].to_numpy(dtype=np.float64)),
        vol,
        **kwargs,
    )
    
    return pd.DataFrame({
        "strength": result[0],
        "shape": result[1],
        "size": result[2],
        "volume_gate": result[3],
        "volume_rel": result[4],
        "direction": result[5],
        "streak": result[6],
        "momentum": result[7],
        "freshness": result[8],
    }, index=data.index)

In [ ]:
result = _candle_strength_v2(open_=df.open.values, close_=df.close.values, high_=df.high.values, low_=df.low.values, volume_=df.volume.values, window_size=1000, short_window=50)
df['strength'] = result[0]
df['size'] = result[2]
df['direction'] = result[5]
df['momentum'] = result[7]

In [ ]:
"""
Wick Absorption Signal v5 — Streaming architecture for 1.6M+ candles.

Target: < 5s for 1.6M candles on warm JIT.

Architecture:
  Instead of O(n × lookback) density computation, we use:
  1. Fenwick-tree rolling quantiles: O(log B) per insert/remove/query
  2. Bucket-based density with exponential decay: O(1) amortized per bar
  3. All signal components computed in a single fused pass
  4. Total cost: O(n × log B) where B = number of price buckets (~4096)
"""

import numpy as np
from numba import njit


# ═══════════════════════════════════════════════════════════════
# Fenwick Tree (Binary Indexed Tree) for Rolling Quantile
# ═══════════════════════════════════════════════════════════════
# Maintains a frequency histogram over B buckets.
# Operations:
#   update(tree, i, delta): O(log B)
#   prefix_sum(tree, i):    O(log B)
#   quantile(tree, B, q, total): O(log² B) via binary search on prefix
#
# For rolling window: ring buffer tracks which bucket to remove.
# ═══════════════════════════════════════════════════════════════

@njit(nogil=True, cache=True)
def _fenwick_update(tree, i, delta):
    """Add delta to position i (1-indexed)."""
    n = tree.shape[0] - 1  # tree is 1-indexed, size = B+1
    while i <= n:
        tree[i] += delta
        i += i & (-i)


@njit(nogil=True, cache=True)
def _fenwick_prefix(tree, i):
    """Sum of tree[1..i]."""
    s = 0
    while i > 0:
        s += tree[i]
        i -= i & (-i)
    return s


@njit(nogil=True, cache=True)
def _fenwick_quantile(tree, B, q, total):
    """Find smallest index i such that prefix_sum(i) >= q * total.
    O(log B) using binary lifting."""
    if total <= 0:
        return -1
    target = q * total
    if target <= 0:
        # Find first non-zero
        for i in range(1, B + 1):
            if _fenwick_prefix(tree, i) > 0:
                return i
        return -1
    pos = 0
    bit_mask = 1
    while bit_mask <= B:
        bit_mask <<= 1
    bit_mask >>= 1
    cum = 0
    while bit_mask > 0:
        npos = pos + bit_mask
        if npos <= B and cum + tree[npos] < target:
            cum += tree[npos]
            pos = npos
        bit_mask >>= 1
    return pos + 1  # 1-indexed bucket


@njit(nogil=True, cache=True)
def _val_to_bucket(val, vmin, bucket_size, B):
    """Map a continuous value to a bucket index [1, B]."""
    b = int((val - vmin) / bucket_size) + 1
    if b < 1:
        b = 1
    if b > B:
        b = B
    return b


@njit(nogil=True, cache=True)
def _bucket_to_val(b, vmin, bucket_size):
    """Map bucket center back to value."""
    return vmin + (b - 0.5) * bucket_size


# ═══════════════════════════════════════════════════════════════
# Bucket-based Density with Exponential Decay
# ═══════════════════════════════════════════════════════════════
# For each price bucket, maintain an exponentially-decayed
# accumulator of weighted wick contributions.
#
# On each bar:
#   1. Decay all buckets by multiplying by exp(-λ)
#      → Instead of decaying all B buckets, track a global
#        decay factor and store un-decayed values. The actual
#        value at bucket b is: raw[b] * global_decay_factor.
#        This makes the per-bar cost O(1) for decay.
#   2. Add new wick contribution to its bucket: O(1)
#   3. Query: sum nearby buckets weighted by gaussian kernel: O(k)
#      where k = number of buckets within 3σ (~6 buckets typically)
#
# Total per bar: O(k) ≈ O(1)
# ═══════════════════════════════════════════════════════════════

@njit(nogil=True, cache=True)
def _density_score_bucketed(
    wick_tips,       # price of wick tip
    wick_abs,        # absolute wick size
    wick_q95,        # rolling p95 of wick_abs
    vol_rel,         # relative volume
    atr,             # ATR
    atr_median,      # median ATR
    base_half_life,
    hl_min, hl_max,
    bw_mult,
    bw_floor,
    price_min, price_max,  # price range for bucketing
    n_buckets,             # number of price buckets
):
    """
    O(n × k) density score where k = buckets within 3σ bandwidth (~6).
    Uses lazy global decay to avoid touching all buckets per bar.
    """
    n = wick_tips.shape[0]
    scores = np.empty(n, dtype=np.float64)
    scores[:] = 0.0

    B = n_buckets
    bucket_size = (price_max - price_min) / B
    if bucket_size < 1e-12:
        return scores

    # Accumulator per bucket: stores un-decayed weighted mass
    buckets = np.zeros(B + 1, dtype=np.float64)  # 1-indexed

    # Global cumulative decay factor
    # actual_value[b] = buckets[b] * global_decay
    # When we add new mass, we add it as mass / global_decay
    # so that actual = (mass / global_decay) * global_decay = mass
    global_decay = 1.0

    ln2 = np.log(2.0)
    MIN_DECAY = 1e-15  # renormalize when global_decay gets too small

    for i in range(n):
        tip_i = wick_tips[i]
        atr_i = atr[i]
        atr_med_i = atr_median[i]
        abs_i = wick_abs[i]
        q95_i = wick_q95[i]

        # Validate
        if np.isnan(tip_i) or np.isnan(atr_i) or atr_i < 1e-12:
            scores[i] = np.nan
            continue
        if np.isnan(atr_med_i) or atr_med_i < 1e-12:
            scores[i] = np.nan
            continue

        # ── Adaptive half-life → per-bar decay ──
        hl = base_half_life * (atr_med_i / atr_i)
        if hl < hl_min:
            hl = hl_min
        elif hl > hl_max:
            hl = hl_max
        decay_per_bar = np.exp(-ln2 / hl)

        # Apply one-bar decay to global factor
        global_decay *= decay_per_bar

        # Renormalize if global_decay is too small
        if global_decay < MIN_DECAY:
            for b in range(1, B + 1):
                buckets[b] *= global_decay
            global_decay = 1.0

        # ── Add current wick's contribution ──
        if not np.isnan(abs_i) and not np.isnan(q95_i) and q95_i > 1e-12:
            rel = abs_i / q95_i
            w_mag = (rel * rel) / (rel * rel + 1.0)

            if w_mag > 0.05:  # same min_wmag filter
                vr = vol_rel[i]
                if np.isnan(vr) or vr < 1e-12:
                    w_vol = 1.0
                else:
                    w_vol = np.sqrt(vr)

                weight = w_mag * w_vol
                b_i = _val_to_bucket(tip_i, price_min, bucket_size, B)
                # Store as un-decayed: actual = (weight/global_decay) * global_decay = weight
                buckets[b_i] += weight / global_decay

        # ── Query: sum nearby buckets with gaussian kernel ──
        bw = bw_mult * atr_i
        if bw < bw_floor:
            bw = bw_floor

        # How many buckets is 3σ?
        radius_buckets = int(3.0 * bw / bucket_size) + 1
        center_bucket = _val_to_bucket(tip_i, price_min, bucket_size, B)

        b_lo = center_bucket - radius_buckets
        if b_lo < 1:
            b_lo = 1
        b_hi = center_bucket + radius_buckets
        if b_hi > B:
            b_hi = B

        inv_bw2 = 1.0 / (2.0 * bw * bw)
        acc = 0.0

        for b in range(b_lo, b_hi + 1):
            raw_mass = buckets[b] * global_decay
            if raw_mass < 1e-15:
                continue
            # Distance from bucket center to tip_i
            bucket_center = price_min + (b - 0.5) * bucket_size
            dist = tip_i - bucket_center
            kernel = np.exp(-dist * dist * inv_bw2)
            acc += raw_mass * kernel

        scores[i] = acc

    return scores


# ═══════════════════════════════════════════════════════════════
# Rolling Quantile via Fenwick Tree
# ═══════════════════════════════════════════════════════════════

@njit(nogil=True, cache=True)
def rolling_quantile_fenwick(data, window, q, B=4096):
    """
    Rolling quantile using Fenwick tree.
    O(n × log B) total. B = number of histogram buckets.
    """
    n = data.shape[0]
    out = np.empty(n, dtype=np.float64)
    out[:] = np.nan

    # Determine range from data
    dmin = np.inf
    dmax = -np.inf
    for i in range(n):
        v = data[i]
        if not np.isnan(v):
            if v < dmin:
                dmin = v
            if v > dmax:
                dmax = v
    if dmin >= dmax:
        dmax = dmin + 1.0

    # Add padding
    rng = dmax - dmin
    dmin -= rng * 0.01
    dmax += rng * 0.01
    bucket_size = (dmax - dmin) / B

    tree = np.zeros(B + 1, dtype=np.int64)
    ring = np.empty(window, dtype=np.int32)  # bucket indices for removal
    ring[:] = -1
    total = 0

    for i in range(n):
        slot = i % window
        v = data[i]

        # Remove leaving element
        if i >= window:
            old_b = ring[slot]
            if old_b >= 0:
                _fenwick_update(tree, old_b, -1)
                total -= 1

        # Insert new
        if np.isnan(v):
            ring[slot] = -1
        else:
            b = _val_to_bucket(v, dmin, bucket_size, B)
            ring[slot] = b
            _fenwick_update(tree, b, 1)
            total += 1

        # Query
        if total >= 10:
            qb = _fenwick_quantile(tree, B, q, total)
            if qb >= 1:
                out[i] = _bucket_to_val(qb, dmin, bucket_size)

    return out


@njit(nogil=True, cache=True)
def rolling_median_fenwick(data, window, B=4096):
    """Rolling median via Fenwick tree. O(n × log B)."""
    return rolling_quantile_fenwick(data, window, 0.5, B)


# ═══════════════════════════════════════════════════════════════
# EMA & ATR (O(n), unchanged)
# ═══════════════════════════════════════════════════════════════

@njit(nogil=True, cache=True)
def rolling_ema(arr, span):
    n = arr.shape[0]
    out = np.empty(n, dtype=np.float64)
    alpha = 2.0 / (span + 1.0)
    last = np.nan
    for i in range(n):
        v = arr[i]
        if np.isnan(v):
            out[i] = last
        elif np.isnan(last):
            last = v
            out[i] = v
        else:
            last = alpha * v + (1.0 - alpha) * last
            out[i] = last
    return out


@njit(nogil=True, cache=True)
def rolling_atr(high, low, close, period):
    n = high.shape[0]
    tr = np.empty(n, dtype=np.float64)
    tr[0] = high[0] - low[0]
    for i in range(1, n):
        hl = high[i] - low[i]
        hc = abs(high[i] - close[i - 1])
        lc = abs(low[i] - close[i - 1])
        tr[i] = max(hl, max(hc, lc))
    return rolling_ema(tr, period)


# ═══════════════════════════════════════════════════════════════
# Normalize (Fenwick-based)
# ═══════════════════════════════════════════════════════════════

@njit(nogil=True, cache=True)
def _normalize_fenwick(scores, window, B=2048):
    """Normalize by rolling p95 using Fenwick quantile."""
    n = scores.shape[0]
    out = np.empty(n, dtype=np.float64)
    out[:] = np.nan

    # Range
    smin = np.inf
    smax = -np.inf
    for i in range(n):
        v = scores[i]
        if not np.isnan(v):
            if v < smin:
                smin = v
            if v > smax:
                smax = v
    if smin >= smax:
        return out

    rng = smax - smin
    smin -= rng * 0.01
    smax += rng * 0.01
    bucket_size = (smax - smin) / B

    tree = np.zeros(B + 1, dtype=np.int64)
    ring = np.empty(window, dtype=np.int32)
    ring[:] = -1
    total = 0

    for i in range(n):
        slot = i % window
        v = scores[i]

        if i >= window:
            old_b = ring[slot]
            if old_b >= 0:
                _fenwick_update(tree, old_b, -1)
                total -= 1

        if np.isnan(v):
            ring[slot] = -1
        else:
            b = _val_to_bucket(v, smin, bucket_size, B)
            ring[slot] = b
            _fenwick_update(tree, b, 1)
            total += 1

        if total < 20 or np.isnan(v):
            continue

        qb = _fenwick_quantile(tree, B, 0.95, total)
        if qb >= 1:
            q95 = _bucket_to_val(qb, smin, bucket_size)
            if q95 > 1e-12:
                out[i] = v / q95
            else:
                out[i] = 0.0

    return out


# ═══════════════════════════════════════════════════════════════
# Fused Signal Computation (single pass)
# ═══════════════════════════════════════════════════════════════

@njit(nogil=True, cache=True)
def _compute_signals(
    buy_norm, sell_norm,
    low, high, atr, atr_median,
    persistence_window,
    reaction_window,
):
    """
    Single-pass computation of intensity, persistence, reaction,
    signal, total, skew. Uses Fenwick trees for persistence medians.
    """
    n = buy_norm.shape[0]
    buy_int = np.empty(n, dtype=np.float64)
    buy_per = np.empty(n, dtype=np.float64)
    buy_rea = np.empty(n, dtype=np.float64)
    buy_sig = np.empty(n, dtype=np.float64)
    sell_int = np.empty(n, dtype=np.float64)
    sell_per = np.empty(n, dtype=np.float64)
    sell_rea = np.empty(n, dtype=np.float64)
    sell_sig = np.empty(n, dtype=np.float64)
    abs_total = np.empty(n, dtype=np.float64)
    abs_skew = np.empty(n, dtype=np.float64)

    k2 = 2.25  # 1.5^2
    pw = persistence_window
    rw = reaction_window

    # Persistence: use simple EMA-based median approximation
    # instead of exact median to keep O(1) per bar.
    # Approximation: track running median via "median filter" trick:
    # median += sign(x - median) * step_size
    # step_size adapts to be ~1% of the local range.
    buy_run_med = 0.0
    sell_run_med = 0.0
    buy_streak = 0.0
    sell_streak = 0.0
    buy_med_init = False
    sell_med_init = False

    for i in range(n):
        atr_i = atr[i]
        atr_med_i = atr_median[i]
        bn = buy_norm[i]
        sn = sell_norm[i]

        # ── INTENSITY ──
        if np.isnan(bn) or bn <= 0.0:
            buy_int[i] = 0.0
        else:
            buy_int[i] = (bn * bn) / (bn * bn + k2)

        if np.isnan(sn) or sn <= 0.0:
            sell_int[i] = 0.0
        else:
            sell_int[i] = (sn * sn) / (sn * sn + k2)

        # ── PERSISTENCE (running median approximation) ──
        # Adaptive step: converges in ~50 bars
        if not np.isnan(bn) and bn > 0:
            if not buy_med_init:
                buy_run_med = bn
                buy_med_init = True
            else:
                step = max(buy_run_med * 0.02, 1e-6)
                if bn > buy_run_med:
                    buy_run_med += step
                else:
                    buy_run_med -= step

        if np.isnan(bn) or np.isnan(atr_i) or np.isnan(atr_med_i) or not buy_med_init:
            buy_streak = 0.0
            buy_per[i] = 0.0
        else:
            if bn > buy_run_med and buy_run_med > 0:
                buy_streak += 1.0
            else:
                buy_streak *= 0.5
            if atr_i < 1e-12:
                req = pw * 0.5
            else:
                rr = atr_med_i / atr_i
                req = pw * 0.3 * rr
                if req < 3.0:
                    req = 3.0
                elif req > pw * 0.8:
                    req = pw * 0.8
            ratio = buy_streak / req
            buy_per[i] = (ratio * ratio) / (ratio * ratio + 1.0)

        if not np.isnan(sn) and sn > 0:
            if not sell_med_init:
                sell_run_med = sn
                sell_med_init = True
            else:
                step = max(sell_run_med * 0.02, 1e-6)
                if sn > sell_run_med:
                    sell_run_med += step
                else:
                    sell_run_med -= step

        if np.isnan(sn) or np.isnan(atr_i) or np.isnan(atr_med_i) or not sell_med_init:
            sell_streak = 0.0
            sell_per[i] = 0.0
        else:
            if sn > sell_run_med and sell_run_med > 0:
                sell_streak += 1.0
            else:
                sell_streak *= 0.5
            if atr_i < 1e-12:
                req = pw * 0.5
            else:
                rr = atr_med_i / atr_i
                req = pw * 0.3 * rr
                if req < 3.0:
                    req = 3.0
                elif req > pw * 0.8:
                    req = pw * 0.8
            ratio = sell_streak / req
            sell_per[i] = (ratio * ratio) / (ratio * ratio + 1.0)

        # ── REACTION (OLS over short window) ──
        if np.isnan(atr_i) or atr_i < 1e-12:
            buy_rea[i] = 0.0
            sell_rea[i] = 0.0
        else:
            jstart = i - rw + 1
            if jstart < 0:
                jstart = 0

            # Buy (lows)
            cnt = 0
            sx = 0.0; sy = 0.0; sxx = 0.0; sxy = 0.0
            for j in range(jstart, i + 1):
                tip = low[j]
                if not np.isnan(tip):
                    x = float(j - jstart)
                    sx += x; sy += tip; sxx += x * x; sxy += x * tip
                    cnt += 1
            if cnt < 3:
                buy_rea[i] = 0.5
            else:
                denom = cnt * sxx - sx * sx
                slope = 0.0 if abs(denom) < 1e-12 else (cnt * sxy - sx * sy) / denom
                directed = slope / atr_i
                if directed >= 0:
                    g = np.exp(-0.5 * (directed / 0.08) ** 2)
                    bonus = directed * 5.0
                    if bonus > 0.05: bonus = 0.05
                    val = 0.95 * g + bonus
                    if val > 1.0: val = 1.0
                    buy_rea[i] = val
                else:
                    g = np.exp(-0.5 * (directed / 0.025) ** 2)
                    val = 0.95 * g
                    if val < 0.02: val = 0.02
                    buy_rea[i] = val

            # Sell (highs)
            cnt = 0
            sx = 0.0; sy = 0.0; sxx = 0.0; sxy = 0.0
            for j in range(jstart, i + 1):
                tip = high[j]
                if not np.isnan(tip):
                    x = float(j - jstart)
                    sx += x; sy += tip; sxx += x * x; sxy += x * tip
                    cnt += 1
            if cnt < 3:
                sell_rea[i] = 0.5
            else:
                denom = cnt * sxx - sx * sx
                slope = 0.0 if abs(denom) < 1e-12 else (cnt * sxy - sx * sy) / denom
                directed = -(slope / atr_i)
                if directed >= 0:
                    g = np.exp(-0.5 * (directed / 0.08) ** 2)
                    bonus = directed * 5.0
                    if bonus > 0.05: bonus = 0.05
                    val = 0.95 * g + bonus
                    if val > 1.0: val = 1.0
                    sell_rea[i] = val
                else:
                    g = np.exp(-0.5 * (directed / 0.025) ** 2)
                    val = 0.95 * g
                    if val < 0.02: val = 0.02
                    sell_rea[i] = val

        # ── COMBINE + SKEW ──
        b = buy_int[i] * buy_per[i] * buy_rea[i]
        s = sell_int[i] * sell_per[i] * sell_rea[i]
        buy_sig[i] = b
        sell_sig[i] = s
        abs_total[i] = b + s
        abs_skew[i] = (b - s) / (b + s + 1e-12)

    return (buy_int, buy_per, buy_rea, buy_sig,
            sell_int, sell_per, sell_rea, sell_sig,
            abs_total, abs_skew)


# ═══════════════════════════════════════════════════════════════
# Public API
# ═══════════════════════════════════════════════════════════════

def wick_absorption_signal(
    data,
    atr_period=50,
    atr_median_window=500,
    lookback=500,
    base_half_life=20,
    hl_min=10,
    hl_max=400,
    bw_mult=0.15,
    norm_window=2000,
    tick_size=0.25,
    tick_bw_mult=3.0,
    vol_median_window=100,
    wick_mag_quantile=0.95,
    wick_mag_window=500,
    persistence_window=50,
    reaction_window=8,
    n_price_buckets=4096,
):
    """
    Full pipeline: clustering → signal → skew.

    Streaming architecture — O(n × log B) total:
      - Fenwick-tree rolling quantiles
      - Bucket-based density with lazy exponential decay
      - Fused single-pass signal computation

    New param:
        n_price_buckets: resolution of price bucketing for density (4096).
    """
    # ── Extract arrays ──
    high = np.ascontiguousarray(data["high"].to_numpy(dtype=np.float64))
    low = np.ascontiguousarray(data["low"].to_numpy(dtype=np.float64))
    close = np.ascontiguousarray(data["close"].to_numpy(dtype=np.float64))
    open_ = np.ascontiguousarray(data["open"].to_numpy(dtype=np.float64))
    n = len(close)

    has_volume = "volume" in data.columns
    if has_volume:
        vol_raw = np.ascontiguousarray(data["volume"].to_numpy(dtype=np.float64))
        vol_med = rolling_median_fenwick(vol_raw, vol_median_window)
        vol_rel = np.empty(n, dtype=np.float64)
        for i in range(n):
            if not np.isnan(vol_med[i]) and vol_med[i] > 1e-12:
                vol_rel[i] = vol_raw[i] / vol_med[i]
            else:
                vol_rel[i] = 1.0
        vol_rel = np.ascontiguousarray(vol_rel)
    else:
        vol_rel = np.ones(n, dtype=np.float64)

    # ── ATR ──
    atr = rolling_atr(high, low, close, atr_period)
    atr_med = rolling_median_fenwick(atr, atr_median_window)

    # ── Wick arrays ──
    body_bottom = np.minimum(open_, close)
    body_top = np.maximum(open_, close)
    lower_wick_abs = np.ascontiguousarray(body_bottom - low)
    upper_wick_abs = np.ascontiguousarray(high - body_top)
    lower_tips = np.ascontiguousarray(low.copy())
    upper_tips = np.ascontiguousarray(high.copy())

    # ── Rolling quantile of wick magnitude ──
    lower_q = rolling_quantile_fenwick(lower_wick_abs, wick_mag_window, wick_mag_quantile)
    upper_q = rolling_quantile_fenwick(upper_wick_abs, wick_mag_window, wick_mag_quantile)

    # ── Price range for bucketing ──
    pmin = np.nanmin(low)
    pmax = np.nanmax(high)
    prng = pmax - pmin
    pmin -= prng * 0.02
    pmax += prng * 0.02

    bw_floor = tick_size * tick_bw_mult

    # ── Density scores (bucketed) ──
    buy_raw = _density_score_bucketed(
        lower_tips, lower_wick_abs, lower_q, vol_rel,
        atr, atr_med,
        base_half_life, hl_min, hl_max,
        bw_mult, bw_floor,
        pmin, pmax, n_price_buckets,
    )
    sell_raw = _density_score_bucketed(
        upper_tips, upper_wick_abs, upper_q, vol_rel,
        atr, atr_med,
        base_half_life, hl_min, hl_max,
        bw_mult, bw_floor,
        pmin, pmax, n_price_buckets,
    )

    # ── Normalize ──
    buy_norm = _normalize_fenwick(buy_raw, norm_window)
    sell_norm = _normalize_fenwick(sell_raw, norm_window)

    # ── Signals ──
    (buy_int, buy_per, buy_rea, buy_sig,
     sell_int, sell_per, sell_rea, sell_sig,
     abs_total, abs_skew_arr) = _compute_signals(
        buy_norm, sell_norm,
        low, high, atr, atr_med,
        persistence_window, reaction_window,
    )

    # ── Output ──
    data = data.copy()
    data["buy_pressure"] = buy_raw
    data["sell_pressure"] = sell_raw
    data["buy_pressure_norm"] = buy_norm
    data["sell_pressure_norm"] = sell_norm
    data["buy_intensity"] = buy_int
    data["buy_persistence"] = buy_per
    data["buy_reaction"] = buy_rea
    data["buy_signal"] = buy_sig
    data["sell_intensity"] = sell_int
    data["sell_persistence"] = sell_per
    data["sell_reaction"] = sell_rea
    data["sell_signal"] = sell_sig
    data["absorption_total"] = abs_total
    data["absorption_skew"] = abs_skew_arr

    return data






In [ ]:
# ─────────────────────────────────────────────────────────────
# Uso
# ─────────────────────────────────────────────────────────────
#
df = wick_absorption_signal(df)
#
# Interpretação:
#   buy_pressure_norm > 1.0  →  cluster de lower wicks acima do p95
#                                → absorção compradora significativa
#   sell_pressure_norm > 1.0 →  cluster de upper wicks acima do p95
#                                → absorção vendedora significativa
#
# Para plotar sobre candlestick:
#   - Use buy_pressure_norm como heatmap/cor nos candles
#   - Ou aplique um limiar visual (ex: > 1.5) para marcar

In [ ]:
df

In [21]:
"""
Candle Force Indicator — Protótipo v0.4
========================================
Mudanças vs v0.3:
- Normalização: troca rank (CDF empírica) por tanh adaptativo.
  normalized = tanh(raw / q95_|raw|)
  O q95 do |force_raw| rolling define a escala do tanh.
  Preserva proporcionalidade: candle pequeno → valor pequeno.
  Extremos comprimidos suavemente, não achatados.
- Remove _rolling_rank_normalize.

Filosofia:
- q95 como unidade de medida em tudo — inclusive na normalização final.
- Shadow é força direcional.
- Zero magic numbers.
"""

import numpy as np
import pandas as pd
from numba import njit


# ─────────────────────────────────────────────────────────
# Primitivas
# ─────────────────────────────────────────────────────────

@njit(nogil=True, cache=False)
def _bisect_left(a, n, x):
    lo = 0
    hi = n
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] < x:
            lo = mid + 1
        else:
            hi = mid
    return lo


@njit(nogil=True, cache=False)
def _rolling_quantile(arr, window, q):
    """Rolling quantile incremental, causal, exato."""
    n = arr.shape[0]
    out = np.empty(n, dtype=np.float64)
    out[:] = np.nan

    if window <= 0:
        return out

    sorted_vals = np.empty(window, dtype=np.float64)
    ring = np.empty(window, dtype=np.float64)
    ring[:] = np.nan
    m = 0

    for i in range(n):
        slot = i % window

        if i >= window:
            old_v = ring[slot]
            if not np.isnan(old_v):
                pos = _bisect_left(sorted_vals, m, old_v)
                for j in range(pos, m - 1):
                    sorted_vals[j] = sorted_vals[j + 1]
                m -= 1

        new_v = arr[i]
        ring[slot] = new_v
        if not np.isnan(new_v):
            pos = _bisect_left(sorted_vals, m, new_v)
            for j in range(m, pos, -1):
                sorted_vals[j] = sorted_vals[j - 1]
            sorted_vals[pos] = new_v
            m += 1

        if i >= window - 1 and m == window:
            if m == 0:
                out[i] = np.nan
            elif q <= 0.0:
                out[i] = sorted_vals[0]
            elif q >= 1.0:
                out[i] = sorted_vals[m - 1]
            else:
                idx_f = q * (m - 1)
                lo_i = int(np.floor(idx_f))
                hi_i = int(np.ceil(idx_f))
                if lo_i == hi_i:
                    out[i] = sorted_vals[lo_i]
                else:
                    frac = idx_f - lo_i
                    out[i] = sorted_vals[lo_i] + (sorted_vals[hi_i] - sorted_vals[lo_i]) * frac

    return out


@njit(nogil=True, cache=False)
def _adaptive_tanh(arr, scale):
    """
    tanh(arr / scale) onde scale é um array rolling.
    Preserva proporcionalidade na região linear, comprime extremos.
    """
    n = arr.shape[0]
    out = np.empty(n, dtype=np.float64)
    out[:] = np.nan
    eps = 1e-12

    for i in range(n):
        if np.isnan(arr[i]):
            continue
        if np.isnan(scale[i]):
            continue
        s = scale[i]
        if s < eps:
            s = eps
        out[i] = np.tanh(arr[i] / s)

    return out


@njit(nogil=True, cache=False)
def _atr(high, low, close, period):
    """ATR Wilder (EMA-style)."""
    n = high.shape[0]
    out = np.empty(n, dtype=np.float64)
    out[:] = np.nan

    if n == 0:
        return out
    if period < 1:
        return out

    tr = np.empty(n, dtype=np.float64)
    tr[0] = high[0] - low[0]
    for i in range(1, n):
        hl = high[i] - low[i]
        hc = abs(high[i] - close[i - 1])
        lc = abs(low[i] - close[i - 1])
        tr_i = hl
        if hc > tr_i:
            tr_i = hc
        if lc > tr_i:
            tr_i = lc
        tr[i] = tr_i

    if n < period:
        return out

    seed = 0.0
    for i in range(period):
        seed += tr[i]
    out[period - 1] = seed / period

    for i in range(period, n):
        out[i] = ((out[i - 1] * (period - 1.0)) + tr[i]) / period

    return out


# ─────────────────────────────────────────────────────────
# Core
# ─────────────────────────────────────────────────────────

@njit(nogil=True, cache=False)
def _candle_force_core(
    open_,
    close_,
    high_,
    low_,
    atr_short,
    atr_long,
    q95_body,
    q95_shdn,
    q95_shup,
):
    """
    Net force usando scores normalizados por q95.

    Retorna 5 arrays:
      force_short_raw  — net force q95-weighted, ajustado vol curto
      force_long_raw   — net force q95-weighted, ajustado vol longo
      body_score       — body / q95_body
      rej_dn_score     — shdn / q95_shdn
      rej_up_score     — shup / q95_shup
    """
    n = close_.shape[0]
    force_short_raw = np.empty(n, dtype=np.float64)
    force_long_raw = np.empty(n, dtype=np.float64)
    body_score = np.empty(n, dtype=np.float64)
    rej_dn_score = np.empty(n, dtype=np.float64)
    rej_up_score = np.empty(n, dtype=np.float64)

    force_short_raw[:] = np.nan
    force_long_raw[:] = np.nan
    body_score[:] = np.nan
    rej_dn_score[:] = np.nan
    rej_up_score[:] = np.nan

    eps = 1e-12

    for i in range(n):
        if (np.isnan(atr_long[i]) or np.isnan(atr_short[i])
                or np.isnan(q95_body[i]) or np.isnan(q95_shdn[i])
                or np.isnan(q95_shup[i])):
            continue

        o = open_[i]
        c = close_[i]
        h = high_[i]
        l = low_[i]

        body = abs(c - o)
        shdn = min(c, o) - l
        shup = h - max(c, o)

        # Direção
        if c > o:
            direction = 1.0
        elif c < o:
            direction = -1.0
        else:
            direction = 0.0

        # Scores em unidades q95
        q_b = q95_body[i]
        q_d = q95_shdn[i]
        q_u = q95_shup[i]
        if q_b < eps:
            q_b = eps
        if q_d < eps:
            q_d = eps
        if q_u < eps:
            q_u = eps

        b_s = body / q_b
        d_s = shdn / q_d
        u_s = shup / q_u

        body_score[i] = b_s
        rej_dn_score[i] = d_s
        rej_up_score[i] = u_s

        # Net force em unidades q95
        net_q95 = direction * b_s + d_s - u_s

        # Ajuste por regime de volatilidade
        atr_s = atr_short[i]
        atr_l = atr_long[i]
        if atr_s < eps:
            atr_s = eps
        if atr_l < eps:
            atr_l = eps

        # force_short: penalizado quando vol local alta
        force_short_raw[i] = net_q95 * (atr_l / atr_s)

        # force_long: penalizado quando vol local baixa
        force_long_raw[i] = net_q95 * (atr_s / atr_l)

    return force_short_raw, force_long_raw, body_score, rej_dn_score, rej_up_score


# ─────────────────────────────────────────────────────────
# API pública
# ─────────────────────────────────────────────────────────

def candle_force(
    df: pd.DataFrame,
    atr_short_period: int = 20,
    atr_long_period: int = 200,
    quantile_window: int = 1000,
    quantile_level: float = 0.95,
    open_col: str = "open",
    high_col: str = "high",
    low_col: str = "low",
    close_col: str = "close",
) -> pd.DataFrame:
    """
    Calcula o Candle Force Indicator.

    Normalização final: tanh(raw / q95_|raw|)
    - raw / q95 = 1.0 → tanh(1) ≈ 0.76  (limiar do anômalo)
    - raw / q95 = 0.3 → tanh(0.3) ≈ 0.29 (pequeno parece pequeno)
    - raw / q95 = 2.0 → tanh(2) ≈ 0.96  (extremo, comprimido suavemente)

    Retorna DataFrame com:
        candle_force_short  — [-1, 1] sensibilidade local
        candle_force_long   — [-1, 1] perspectiva macro
        force_short_raw     — net force bruto (unidades q95, ajustado vol)
        force_long_raw      — net force bruto (unidades q95, ajustado vol)
        body_score          — body / q95_body
        rejection_dn_score  — shdn / q95_shdn
        rejection_up_score  — shup / q95_shup
    """
    _atr_sp = int(atr_short_period)
    _atr_lp = int(atr_long_period)
    _qw = int(quantile_window)
    _ql = float(quantile_level)

    o = df[open_col].to_numpy(dtype=np.float64)
    h = df[high_col].to_numpy(dtype=np.float64)
    l = df[low_col].to_numpy(dtype=np.float64)
    c = df[close_col].to_numpy(dtype=np.float64)

    # ATRs
    atr_s = _atr(h, l, c, _atr_sp)
    atr_l = _atr(h, l, c, _atr_lp)

    # Componentes do candle
    body = np.abs(c - o)
    shdn = np.minimum(c, o) - l
    shup = h - np.maximum(c, o)

    # Quantis rolling — unidades de medida dos componentes
    q_body = _rolling_quantile(body, _qw, _ql)
    q_shdn = _rolling_quantile(shdn, _qw, _ql)
    q_shup = _rolling_quantile(shup, _qw, _ql)

    # Core — force bruto em unidades q95
    f_short_raw, f_long_raw, b_score, rd_score, ru_score = _candle_force_core(
        o, c, h, l,
        atr_s, atr_l,
        q_body, q_shdn, q_shup,
    )

    # Escala adaptativa para o tanh: q95 do |force_raw|
    abs_f_short = np.abs(f_short_raw)
    abs_f_long = np.abs(f_long_raw)
    scale_short = _rolling_quantile(abs_f_short, _qw, _ql)
    scale_long = _rolling_quantile(abs_f_long, _qw, _ql)

    # Normalização: tanh(raw / scale) → [-1, 1]
    f_short_norm = _adaptive_tanh(f_short_raw, scale_short)
    f_long_norm = _adaptive_tanh(f_long_raw, scale_long)

    return pd.DataFrame(
        {
            "candle_force_short": f_short_norm,
            "candle_force_long": f_long_norm,
            "force_short_raw": f_short_raw,
            "force_long_raw": f_long_raw,
            "body_score": b_score,
            "rejection_dn_score": rd_score,
            "rejection_up_score": ru_score,
        },
        index=df.index,
    )



In [31]:
"""
HMM Regime Detection via Candle Force Features — Protótipo
============================================================
Objetivo: verificar se as features do Candle Force Indicator
contêm informação estrutural suficiente para que um HMM
encontre regimes interpretáveis.

Abordagem:
- Usa GaussianHMM do hmmlearn.
- Features: force_short, force_long, body_score, rej_dn_score, rej_up_score.
- Testa 2, 3 e 4 estados.
- Para cada configuração, mostra:
  - Médias e stds de cada feature por estado.
  - Matriz de transição.
  - Distribuição temporal dos estados.
  - Interpretação automática baseada nas médias.

Dados: sintéticos com regimes injetados para validação.
"""

import numpy as np
import pandas as pd
from hmmlearn.hmm import GaussianHMM



def fit_hmm(features, n_states, n_fits=10):
    """
    Fita HMM com múltiplos restarts e retorna o melhor (maior log-likelihood).
    """
    best_model = None
    best_score = -np.inf

    for i in range(n_fits):
        model = GaussianHMM(
            n_components=n_states,
            covariance_type="full",
            n_iter=200,
            random_state=i * 42,
            tol=1e-4,
        )
        try:
            model.fit(features)
            score = model.score(features)
            if score > best_score:
                best_score = score
                best_model = model
        except Exception:
            continue

    return best_model, best_score


def interpret_state(means, feature_names):
    """
    Dá um label interpretável ao estado baseado nas médias das features.
    """
    d = dict(zip(feature_names, means))

    body = d.get("body_score", 0)
    rej_dn = d.get("rej_dn_score", 0)
    rej_up = d.get("rej_up_score", 0)
    f_short = d.get("force_short", 0)

    total_rejection = rej_dn + rej_up
    directional_strength = abs(f_short)

    if body > 0.8 and directional_strength > 0.3:
        if f_short > 0:
            return "IMPULSO BULLISH"
        else:
            return "IMPULSO BEARISH"
    elif total_rejection > 1.2 and body < 0.6:
        return "REJEIÇÃO/CHOPPY"
    elif body < 0.4 and total_rejection < 0.6:
        return "QUIETO/MORTO"
    elif body > 0.6:
        return "IMPULSO MISTO"
    elif total_rejection > 0.8:
        return "REJEIÇÃO MODERADA"
    else:
        return "TRANSIÇÃO"


def analyze_hmm(df, features_df, true_regime=None, n_states_list=[2, 3, 4]):
    """
    Fita e analisa HMM para diferentes números de estados.
    true_regime: array opcional com regime verdadeiro (só para dados sintéticos).
    """
    feature_names = ["force_short", "force_long", "body_score", "rej_dn_score", "rej_up_score"]

    # Prepara features: dropna e escala
    feat_cols = {
        "force_short": "candle_force_short",
        "force_long": "candle_force_long",
        "body_score": "body_score",
        "rej_dn_score": "rejection_dn_score",
        "rej_up_score": "rejection_up_score",
    }

    features_renamed = features_df[list(feat_cols.values())].copy()
    features_renamed.columns = list(feat_cols.keys())

    # Drop NaN
    valid_mask = features_renamed.notna().all(axis=1)
    features_clean = features_renamed[valid_mask].copy()
    true_regime_clean = None
    if true_regime is not None:
        true_regime_clean = true_regime[valid_mask.values]

    X = features_clean.values

    print(f"Features shape: {X.shape}")
    print(f"Feature ranges:")
    for j, name in enumerate(feature_names):
        col = X[:, j]
        print(f"  {name:20s}: [{col.min():+.3f}, {col.max():+.3f}]  "
              f"mean={col.mean():+.3f}  std={col.std():.3f}")

    for n_states in n_states_list:
        print(f"\n{'='*74}")
        print(f"HMM com {n_states} estados")
        print(f"{'='*74}")

        model, score = fit_hmm(X, n_states)

        if model is None:
            print("  FALHOU — não convergiu em nenhum restart.")
            continue

        states = model.predict(X)
        print(f"\nLog-likelihood: {score:.1f}")
        print(f"BIC approx: {-2*score + n_states*(n_states-1 + 2*len(feature_names))*np.log(len(X)):.1f}")

        # Médias e interpretação por estado
        print(f"\n{'Estado':<8} {'Label':<22} " + "  ".join(f"{n:>14s}" for n in feature_names))
        print("-" * 100)

        state_labels = {}
        for s in range(n_states):
            means = model.means_[s]
            label = interpret_state(means, feature_names)
            state_labels[s] = label
            means_str = "  ".join(f"{m:+14.4f}" for m in means)
            print(f"  {s:<6d} {label:<22s} {means_str}")

        # Desvios padrão (diagonal da covariância)
        print(f"\n{'Estado':<8} {'Label':<22} " + "  ".join(f"{'std_'+n:>14s}" for n in feature_names))
        print("-" * 100)
        for s in range(n_states):
            stds = np.sqrt(np.diag(model.covars_[s]))
            stds_str = "  ".join(f"{st:14.4f}" for st in stds)
            print(f"  {s:<6d} {state_labels[s]:<22s} {stds_str}")

        # Matriz de transição
        print(f"\nMatriz de transição:")
        header = "        " + "  ".join(f"  →{s}" for s in range(n_states))
        print(header)
        for s in range(n_states):
            row = "  ".join(f"{model.transmat_[s, t]:.3f}" for t in range(n_states))
            print(f"  {s} ({state_labels[s][:8]:8s})  {row}")

        # Distribuição temporal
        print(f"\nDistribuição dos estados:")
        for s in range(n_states):
            count = (states == s).sum()
            pct = count / len(states) * 100
            bar = "#" * int(pct / 2)
            print(f"  Estado {s} ({state_labels[s]:<22s}): {pct:5.1f}%  {bar}")

        # Concordância com regime verdadeiro (só para dados sintéticos)
        if true_regime_clean is not None:
            print(f"\nConcordância com regimes sintéticos injetados:")
            print(f"  (Regime 0=trending, 1=choppy, 2=quiet)")
            cross = pd.crosstab(
                pd.Series(true_regime_clean, name="true"),
                pd.Series(states, name="hmm"),
                margins=True,
                normalize="index",
            )
            print(cross.round(3).to_string())

        # Duração média dos estados (em barras)
        print(f"\nDuração média por estado (barras consecutivas):")
        durations = {s: [] for s in range(n_states)}
        current_state = states[0]
        current_len = 1
        for i in range(1, len(states)):
            if states[i] == current_state:
                current_len += 1
            else:
                durations[current_state].append(current_len)
                current_state = states[i]
                current_len = 1
        durations[current_state].append(current_len)

        for s in range(n_states):
            if durations[s]:
                d = np.array(durations[s])
                print(f"  Estado {s} ({state_labels[s]:<22s}): "
                      f"mean={d.mean():.1f}  median={np.median(d):.0f}  "
                      f"max={d.max()}  segments={len(d)}")



In [22]:
import numpy as np
import pandas as pd
from lightweight_charts import JupyterChart

df = pd.read_parquet('/Users/marcelogarcia/Documents/Development/autobt/autobt/data/usatec2017_2025.parquet')
df_candle_force = candle_force(df=df)

In [ ]:

analyze_hmm(df, df_candle_force, true_regime=None, n_states_list=[2, 3, 4])

Features shape: (1605646, 5)
Feature ranges:
  force_short         : [-1.000, +1.000]  mean=-0.004  std=0.418
  force_long          : [-1.000, +1.000]  mean=-0.001  std=0.333
  body_score          : [+0.000, +38.018]  mean=+0.330  std=0.411
  rej_dn_score        : [+0.000, +30.091]  mean=+0.316  std=0.413
  rej_up_score        : [+0.000, +20.625]  mean=+0.318  std=0.406

HMM com 2 estados


In [23]:
df = df.merge(df_candle_force, how='left', left_index=True, right_index=True)

In [24]:
df

,open,high,low,close,volume,candle_force_short,candle_force_long,force_short_raw,force_long_raw,body_score,rejection_dn_score,rejection_up_score
2017-01-03 07:00:00+00:00,4895.7,4895.7,4895.4,4895.6,0.00338,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-01-03 07:02:00+00:00,4895.4,4896.4,4895.4,4896.2,0.00117,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-01-03 07:04:00+00:00,4896.1,4896.2,4896.0,4896.2,0.00104,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-01-03 07:06:00+00:00,4896.1,4897.2,4896.0,4896.1,0.00273,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-01-03 07:08:00+00:00,4896.2,4896.5,4896.1,4896.1,0.00065,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-19 16:55:00+00:00,25284.8,25290.0,25271.6,25272.6,0.03770,-0.852236,-0.655030,-1.486433,-1.680024,0.884058,0.156250,0.852459
2025-12-19 16:56:00+00:00,25272.6,25277.2,25266.1,25273.0,0.03500,0.279971,0.173540,0.337539,0.375652,0.028986,1.015625,0.688525
2025-12-19 16:57:00+00:00,25273.0,25276.9,25268.2,25270.2,0.03220,-0.409221,-0.251270,-0.510062,-0.550183,0.202899,0.312500,0.639344
2025-12-19 16:58:00+00:00,25271.9,25273.5,25263.6,25265.9,0.03430,-0.273015,-0.160520,-0.328701,-0.346951,0.434783,0.359375,0.262295


In [ ]:

df['atr_14'] = atr(df)
df['atr_200'] = atr(df, 200)
q95 = rolling_quantile_incremental(df['shdn'].to_numpy(np.float64), 10000, 0.95)
df['q95'] = q95
df['shdn_scores1'] = np.maximum(0.0, df['shdn'] - q95) / df['atr_14']
df['shdn_scores2'] = np.maximum(0.0, df['shdn'] - q95) / df['atr_200']
df['range'] = np.abs(df.open - df.close)



In [25]:
df_plot = df.copy()
if 'time' not in df_plot.columns:
    df_plot = df_plot.rename_axis('time').reset_index()

df_plot['time'] = pd.to_datetime(df_plot['time'], utc=True, errors='coerce').dt.tz_convert(None)
df_plot = df_plot.dropna(subset=['time']).sort_values('time').drop_duplicates('time', keep='last')



In [26]:
from lightweight_charts import Chart, JupyterChart
chart = Chart(height=750, width=1000)
chart.set(    
    df_plot[-2000:-1],
    engine='duckdb',
    engine_options={
        'initial_bars': 3000,
        'chunk_bars': 1500,
        'prefetch_bars': 300,
        'max_bars': 20000,
    },
    
    indicators={
        'candle_force_short': {'pane': 'cs', 'type': 'histogram'},
        'candle_force_long': {'pane': 'cs1', 'type': 'histogram'}
      
        
        
    }
          
)
chart.show(block=True)

/Users/marcelogarcia/Documents/Development/lightweight-charts-python/lightweight_charts/chart.py:213: RuntimeWarning: Chart.show(block=True) was called inside an active event loop. Starting chart listener as a background task; use await chart.show_async() for notebook-friendly blocking behavior.
  warnings.warn(


<Task pending name='Task-7' coro=<Chart.show_async() running at /Users/marcelogarcia/Documents/Development/lightweight-charts-python/lightweight_charts/chart.py:222>>

#### Outros

In [ ]:
import lightweight_charts, inspect
from lightweight_charts import Chart
print(lightweight_charts.__file__)
print(inspect.signature(Chart.set))

In [ ]:
#!pip install -e /Users/marcelogarcia/Documents/Development/lightweight-charts-python
!pip install duckdb

In [ ]:
df[['open','high','low','close']].isna().any(axis=1).sum()
